# R1：持续时间与非平稳概率预测

研究状态：开发机制验证，尚未获得 SOTA。主方案见仓库根目录 `RESEARCH.md` §4.0。

当前已完成原子机制、分布传播、全变量强基线及连续事件年龄检验；下一步诊断多通道共同事件与滞后关系。仅加入年龄特征、通道特征或经典传播本身不构成论文创新。

近邻：[Renewal](https://arxiv.org/abs/2010.01550)、[TimePrism](https://arxiv.org/abs/2509.19975)、[Neural Markov Jump Processes](https://proceedings.mlr.press/v202/seifner23a.html)。

## 1. 计算与复现合同

所有环境、权重和逐条预测保存在仓库外。设备优先 CUDA → MPS → CPU。当前没有 SOTA 声明。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='preflight', repo_root=repo_root, cache_dir=cache_dir)

{
  "stage": "preflight",
  "device": "mps",
  "pin_memory": false,
  "available_ram_gib": 156.03700256347656,
  "cache": "/Users/mark/.cache/marktsf-research/r1",
  "provenance": {
    "head": "c80482f323e02b8983663c9e1f87a5927698d641",
    "sources": {
      "src/r1/data.py": "8c4cb948f871f2fbcd832cb9bf5c9a240be3cf958b8f9fe37e2d691a1aad7bf7",
      "src/r1/duration.py": "4da1cfa40b29878f8c916253523cac1d4cdded3fd24b3b2b4faac7ce9ba5624c",
      "src/r1/evaluation.py": "1c3c1a71358ddb80190f08f4329cc34084fc2d1bd4d6bc0bb8f182d14e8ac29b",
      "src/r1/models.py": "85e8b7cc3a7ebd4899d40b3fce882d4df2129040fbd575c08d1b6f56318e40bc",
      "src/r1/run.py": "6512a86065fec65136ee97fcf99432b6d5f7d90fc6730e323a802025726363f5",
      "src/r1/workflow.py": "2fad7f83750063b1fff12a8373624d294de3cf2182431c2161726461c9e79229"
    },
    "python": "3.12.14",
    "packages": {
      "numpy": "2.5.2",
      "pandas": "3.0.5",
      "scipy": "1.18.1",
      "scikit-learn": "1.9.0",
      "torch": "2.14.0",

## 2. 因果性与评分核验

未来扰动不能改变较早特征；训练目标不能跨验证边界。重叠 origin 不作为独立重复。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='validate', repo_root=repo_root, cache_dir=cache_dir)

{
  "stage": "validation",
  "checks": [
    "future perturbation leaves earlier features unchanged",
    "training labels do not cross split",
    "state age recurrence",
    "perfect forecast scores zero",
    "paired cluster resampling sanity",
    "simulated two-state occupancy agrees with analytic Markov recurrence"
  ],
  "passed": true
}


## 3. 固定开发池数据审计

按任务名称 SHA256 预分组，仅 hash%10<2 的开发任务；每条序列仅前 70%，内部再作 70/30 时间切分。其余数据保留。

In [3]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
max_tasks = 48
max_variates = 4
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='audit', repo_root=repo_root, cache_dir=cache_dir, max_tasks=max_tasks, max_variates=max_variates)

{
  "stage": "data_audit",
  "selected_tasks": 48,
  "selected_variates": 117,
  "intermittent_training_variates": 0,
  "intermittent_atom_variates": 42,
  "empty_training_variates": 0,
  "split": "hash(task)%10<2; first70% development, first70% of development training",
  "official_test_used": false,
  "sota": false
}


## 4. BOOM 持续时间信息诊断

相同 HGB 预算，比较近期上下文与额外年龄表示。目标是未来 horizon 处是否等于最初 96 点中识别的重复值原子；其物理零语义未知；不是新增预测架构，也不是官方完整 BOOM 概率指标。区间按 dataset 聚类，多个 horizon 为探索性分析。

In [4]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'boom'
max_tasks = 48
max_variates = 4
horizons = [1, 8, 32]
max_origins = 512
seed = 2021
max_iter = 100
threads = 4
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='mechanism', repo_root=repo_root, cache_dir=cache_dir, domain=domain, max_tasks=max_tasks, max_variates=max_variates, horizons=horizons, max_origins=max_origins, seed=seed, max_iter=max_iter, threads=threads)

Fitting boom event probe: 113099 rows, 19 datasets


{
  "stage": "development_event_probe",
  "domain": "boom",
  "config": {
    "domain": "boom",
    "max_tasks": 48,
    "max_variates": 4,
    "max_assets": 24,
    "horizons": [
      1,
      8,
      32
    ],
    "max_origins": 512,
    "seed": 2021,
    "max_iter": 100,
    "threads": 4
  },
  "rows": 113099,
  "datasets": 19,
  "items": 40,
  "skipped": {
    "no_atom_in_initial_calibration": 52,
    "short_or_nonfinite": 10,
    "not_intermittent_on_training_prefix": 15
  },
  "seconds": {
    "context_and_age": 5.047784250004042,
    "context_only": 1.4819805000006454
  },
  "scores": [
    {
      "horizon": 1,
      "metric": "brier",
      "context_only": 0.03512029856138725,
      "context_and_age": 0.03463674300072501,
      "relative_improvement_pct": 1.3768549256978242,
      "paired_delta_age_minus_base": {
        "mean": -0.0004835555606622443,
        "ci95": [
          -0.0013677565966792712,
          0.0002121210529593643
        ],
        "clusters": 19
      

## 5. 金融非平稳迁移诊断

正价格转对数收益；状态为超过当时历史 EWMA 波动率 1.5 倍。预测未来高波动事件，不能据此声称价格 SOTA 或交易 alpha。按共同日历月份聚类；短期块间仍可能相关，区间仅作开发诊断。

In [5]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'price'
max_assets = 24
horizons = [1, 8, 32]
max_origins = 512
seed = 2021
max_iter = 100
threads = 4
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='mechanism', repo_root=repo_root, cache_dir=cache_dir, domain=domain, max_assets=max_assets, horizons=horizons, max_origins=max_origins, seed=seed, max_iter=max_iter, threads=threads)

Fitting price event probe: 99725 rows, 2 datasets


{
  "stage": "development_event_probe",
  "domain": "price",
  "config": {
    "domain": "price",
    "max_tasks": 48,
    "max_variates": 4,
    "max_assets": 24,
    "horizons": [
      1,
      8,
      32
    ],
    "max_origins": 512,
    "seed": 2021,
    "max_iter": 100,
    "threads": 4
  },
  "rows": 99725,
  "datasets": 2,
  "items": 41,
  "skipped": {
    "short_or_nonfinite": 7
  },
  "seconds": {
    "context_and_age": 1.6885837500012713,
    "context_only": 1.4988325410013204
  },
  "scores": [
    {
      "horizon": 1,
      "metric": "brier",
      "context_only": 0.11710632829627048,
      "context_and_age": 0.11696254879610334,
      "relative_improvement_pct": 0.12277688341776784,
      "paired_delta_age_minus_base": {
        "mean": -0.0001437795001671318,
        "ci95": [
          -0.00029964993123435,
          1.1314851126155298e-06
        ],
        "clusters": 16
      }
    },
    {
      "horizon": 1,
      "metric": "logloss",
      "context_only": 0.395

## 6. TimesFM 3.0 实际推理基线

固定模型 revision 的开发小样本推理；比较原始值上的 WQL9、MSE。使用 MPS，保留模型原生预处理。该小样本不能支持排行榜结论。

In [6]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
checkpoint_path = '/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2'
max_tasks = 8
max_variates = 2
horizon = 32
context_length = 512
batch_size = 2
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="foundation", repo_root=repo_root, cache_dir=cache_dir, checkpoint_path=checkpoint_path, max_tasks=max_tasks, max_variates=max_variates, horizon=horizon, context_length=context_length, batch_size=batch_size)

/Users/mark/.cache/marktsf-research/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from local directory


{
  "stage": "development_foundation_baseline",
  "device": "mps",
  "use_symmetric_averaging": false,
  "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2",
  "context": 512,
  "horizon": 32,
  "point_forecast_semantics": "TimesFM3 forecast is the median, not the mean; MSE is median-MSE diagnostic",
  "cases": 10,
  "seconds": 1.4124821670047822,
  "scores": [
    {
      "dataset": "ds-1594-T",
      "item": "1594:v14",
      "timesfm3": {
        "mse": 7.00408984057319e-16,
        "mae": 2.6465241054207667e-08,
        "wql9": 1.8372554253965572e-07,
        "quantile_loss_numerator": 8.468877137346454e-07,
        "absolute_target_denominator": 4.609526264165753,
        "crossing_fraction": 0.0,
        "note": "Development 9-quantile WQL; not official full BOOM aggregation"
      },
      "gaussian_random_walk": {
        "mse": 1.1136111826311685e-17,
        "mae": 3.33708133348

## 7. boom 持续时间 × 幅度 2×2

经典经验半马尔可夫对照：几何/年龄依赖 hazard × 独立/年龄条件幅度。所有臂使用相同训练观测、尺度处理和共同随机数。价格用因果波动率标准化收益并逐步还原价格；这是已有模型机制基线，不是新算法。

训练只计实际观测到的转移，不把末尾右删失当作状态结束。只使用开发数据；MSE 为预测中位数的误差，WQL9 为主诊断。

已修正初始状态段左删失：未达到最高年龄组且起点未知的训练记录不作为精确年龄；达到最高截断年龄后保留。

In [7]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'boom'
max_tasks = 48
max_variates = 4
max_assets = 24
horizon = 32
samples = 512
origins_per_series = 6
seed = 2021
smoothing = 20.0
age_cap = 128
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="factorial", repo_root=repo_root, cache_dir=cache_dir, domain=domain, max_tasks=max_tasks, max_variates=max_variates, max_assets=max_assets, horizon=horizon, samples=samples, origins_per_series=origins_per_series, seed=seed, smoothing=smoothing, age_cap=age_cap)

{
  "domain": "boom",
  "cases": 234,
  "datasets": 18,
  "skipped": {
    "no_atom_in_initial_calibration": 52,
    "missing_training_state": 1,
    "short_or_nonfinite": 10,
    "not_intermittent_on_training_prefix": 15
  },
  "scores": {
    "geometric_independent": {
      "wql9": 8452.370171711937,
      "mse": 1.1124941917947662,
      "mae": 0.4251831454409998
    },
    "age_independent": {
      "wql9": 2405620.2195909387,
      "mse": 1.0971802840856246,
      "mae": 0.4218345712398613
    },
    "geometric_coupled": {
      "wql9": 7592.07590819366,
      "mse": 1.1104962455416825,
      "mae": 0.41592440207784764
    },
    "age_coupled": {
      "wql9": 11564.256049865551,
      "mse": 1.0362094175446668,
      "mae": 0.38274676038891836
    }
  },
  "wql9_effects_negative_is_better": {
    "age_effect_independent": {
      "mean": 2397167.849419227,
      "ci95": [
        -0.03498738183629532,
        7189645.498261582
      ],
      "clusters": 18
    },
    "coupling_e

## 8. price 持续时间 × 幅度 2×2

经典经验半马尔可夫对照：几何/年龄依赖 hazard × 独立/年龄条件幅度。所有臂使用相同训练观测、尺度处理和共同随机数。价格用因果波动率标准化收益并逐步还原价格；这是已有模型机制基线，不是新算法。

训练只计实际观测到的转移，不把末尾右删失当作状态结束。只使用开发数据；MSE 为预测中位数的误差，WQL9 为主诊断。

已修正初始状态段左删失：未达到最高年龄组且起点未知的训练记录不作为精确年龄；达到最高截断年龄后保留。

In [8]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'price'
max_tasks = 48
max_variates = 4
max_assets = 24
horizon = 32
samples = 512
origins_per_series = 6
seed = 2021
smoothing = 20.0
age_cap = 128
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="factorial", repo_root=repo_root, cache_dir=cache_dir, domain=domain, max_tasks=max_tasks, max_variates=max_variates, max_assets=max_assets, horizon=horizon, samples=samples, origins_per_series=origins_per_series, seed=seed, smoothing=smoothing, age_cap=age_cap)

{
  "domain": "price",
  "cases": 246,
  "datasets": 2,
  "skipped": {
    "short_or_nonfinite": 7
  },
  "scores": {
    "geometric_independent": {
      "wql9": 0.045372253293992464,
      "mse": 266.23266908376286,
      "mae": 7.22686086233174
    },
    "age_independent": {
      "wql9": 0.04535653476821272,
      "mse": 266.36492260024164,
      "mae": 7.233903689921366
    },
    "geometric_coupled": {
      "wql9": 0.045242871051369316,
      "mse": 263.5919672239592,
      "mae": 7.201754416840409
    },
    "age_coupled": {
      "wql9": 0.045234815271667124,
      "mse": 266.71684303717046,
      "mae": 7.23767342417029
    }
  },
  "wql9_effects_negative_is_better": {
    "age_effect_independent": {
      "mean": -1.5718525779738247e-05,
      "ci95": null,
      "clusters": 6,
      "reason": "fewer_than_8_clusters"
    },
    "coupling_effect_geometric": {
      "mean": -0.0001293822426231342,
      "ci95": null,
      "clusters": 6,
      "reason": "fewer_than_8_clusters

## 9. 由失败样例触发的尺度反馈消融

上一步 BOOM 的概率尾部出现极端误差，不能因为中位数 MSE 改善就判成功。本步固定原开发池、origin、seed 和四个模型臂，仅冻结预测期尺度，检验递归尺度反馈是否放大尾部。该修改由开发结果触发，属于探索，不是预注册确认试验。原失败结果保留。

In [9]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'boom'
max_tasks = 48
max_variates = 4
horizon = 32
samples = 512
origins_per_series = 6
seed = 2021
smoothing = 20.0
age_cap = 128
scale_feedback = False
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="factorial", repo_root=repo_root, cache_dir=cache_dir, domain=domain, max_tasks=max_tasks, max_variates=max_variates, horizon=horizon, samples=samples, origins_per_series=origins_per_series, seed=seed, smoothing=smoothing, age_cap=age_cap, scale_feedback=scale_feedback)

{
  "domain": "boom",
  "cases": 234,
  "datasets": 18,
  "skipped": {
    "no_atom_in_initial_calibration": 52,
    "missing_training_state": 1,
    "short_or_nonfinite": 10,
    "not_intermittent_on_training_prefix": 15
  },
  "scores": {
    "geometric_independent": {
      "wql9": 0.4814478754185259,
      "mse": 1.091435323416421,
      "mae": 0.4129602179977013
    },
    "age_independent": {
      "wql9": 0.436321209978273,
      "mse": 1.0585488550833948,
      "mae": 0.39874247019011466
    },
    "geometric_coupled": {
      "wql9": 0.46579623066013665,
      "mse": 1.0911201394954935,
      "mae": 0.409854218882497
    },
    "age_coupled": {
      "wql9": 0.4086924130479591,
      "mse": 1.0168097953894473,
      "mae": 0.37682657591695473
    }
  },
  "wql9_effects_negative_is_better": {
    "age_effect_independent": {
      "mean": -0.04512666544025298,
      "ci95": [
        -0.09508179805107796,
        -0.005003346886277123
      ],
      "clusters": 18
    },
    "co

## 10. 完全相同开发案例上的强基础模型比较

在第 9 步预先确定的全部案例上运行 TimesFM 3.0，按 dataset 与 origin 恢复原生多变量输入，保留所选通道。每个臂使用相同目标、horizon 和聚合。经验模型在历史前缀上拟合，TimesFM 保持冻结；此处只判断是否有值得继续的性能缺口。

In [10]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
checkpoint_path = '/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2'
case_scores_path = _["cache"] + "/case_scores.parquet"
domain = 'boom'
max_tasks = 48
max_variates = 4
horizon = 32
context_length = 2048
batch_size = 2
seed = 2021
use_symmetric_averaging = True
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="matched_foundation", repo_root=repo_root, cache_dir=cache_dir, checkpoint_path=checkpoint_path, case_scores_path=case_scores_path, domain=domain, max_tasks=max_tasks, max_variates=max_variates, horizon=horizon, context_length=context_length, batch_size=batch_size, seed=seed, use_symmetric_averaging=use_symmetric_averaging)

Loading weights from local directory
TimesFM3 native multivariate inference: 108 groups / 234 cases


{
  "stage": "matched_development_foundation",
  "domain": "boom",
  "cases": 234,
  "native_groups": 108,
  "context_max": 2048,
  "horizon": 32,
  "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2",
  "reference_sha256": "3c9e5e80bbce5a66a738126543b75d9b88ab18d332ee39dd0700691e2d755acf",
  "use_symmetric_averaging": true,
  "prediction_directory": "/Users/mark/.cache/marktsf-research/r1/foundation_predictions_a71190020118",
  "provenance": {
    "head": "c80482f323e02b8983663c9e1f87a5927698d641",
    "sources": {
      "src/r1/data.py": "8c4cb948f871f2fbcd832cb9bf5c9a240be3cf958b8f9fe37e2d691a1aad7bf7",
      "src/r1/duration.py": "4da1cfa40b29878f8c916253523cac1d4cdded3fd24b3b2b4faac7ce9ba5624c",
      "src/r1/evaluation.py": "1c3c1a71358ddb80190f08f4329cc34084fc2d1bd4d6bc0bb8f182d14e8ac29b",
      "src/r1/models.py": "85e8b7cc3a7ebd4899d40b3fce882d4df2129040fbd575c08d1b6f56318e40bc",

## 11. 强基础分布的历史训练特征

仅在原训练前缀内生成基础模型预测；原开发验证案例和目标保持不变。采用历史原子与尺度，测试后缀不参与。先核验原子混合分布的恒等性、单调性、梯度及独立 Monte Carlo 分位数。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
checkpoint_path = '/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2'
reference_path = '/Users/mark/.cache/marktsf-research/r1/factorial_5e3df48b7f0d/case_scores.parquet'
foundation_predictions = '/Users/mark/.cache/marktsf-research/r1/foundation_predictions_a71190020118'
horizon = 32
train_origins = 24
context_length = 2048
batch_size = 2
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='refinement_data', repo_root=repo_root, cache_dir=cache_dir, checkpoint_path=checkpoint_path, reference_path=reference_path, foundation_predictions=foundation_predictions, horizon=horizon, train_origins=train_origins, context_length=context_length, batch_size=batch_size)

/Users/mark/.cache/marktsf-research/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from local directory


Generating frozen-backbone training forecasts: 897 cases


{
  "stage": "refinement_data",
  "cache": "/Users/mark/.cache/marktsf-research/r1/refinement_data_ab1bcdcdbd84",
  "validation": {
    "passed": true,
    "checks": [
      "identity at zero mixture weight",
      "ordered output quantiles",
      "finite gradients",
      "mixture quantiles match independent Monte Carlo"
    ]
  },
  "identity": {
    "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2",
    "reference_sha": "3c9e5e80bbce5a66a738126543b75d9b88ab18d332ee39dd0700691e2d755acf",
    "horizon": 32,
    "train_origins": 24,
    "context": 2048,
    "code_sha": "06bad3c761c612a838a6d9bf8e5129a99804ddcbf3e9c8a2c819db6c48e194e6",
    "data_sha": "8c4cb948f871f2fbcd832cb9bf5c9a240be3cf958b8f9fe37e2d691a1aad7bf7"
  },
  "training_cases": 897,
  "validation_cases": 234,
  "arrays_sha": "e0d7efb6cec0c120dddf30ee050fd462cfd6298957e15db10bfa3e30ac9e216d",
  "metadata_sha": "f18433837e3

## 12. 可微原子概率调整与持续时间对照

固定 TimesFM 幅度分布，学习非负原子混合权重。比较自由逐 horizon 校准、几何持续时间与年龄依赖半马尔可夫传播；自由头参数略多，是必要强对照。混合分布和半马尔可夫本身均为已知机制，此步尚不证明论文创新。训练损失按 origin 可见尺度归一，避免用未来标签幅度定义训练权重。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/refinement_data_ab1bcdcdbd84'
steps = 600
batch_size = 64
width = 64
learning_rate = 0.002
seed = 2021
age_cap = 128
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step='refinement_train', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, steps=steps, batch_size=batch_size, width=width, learning_rate=learning_rate, seed=seed, age_cap=age_cap)

Refiner semi_markov: 5.6s, 28113 parameters


Refiner geometric: 5.8s, 27203 parameters


Refiner direct: 1.8s, 29088 parameters


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/refinement_fit_f20399c5eca5",
  "device": "mps",
  "steps": 600,
  "seed": 2021,
  "parameters": {
    "semi_markov": 28113,
    "geometric": 27203,
    "direct": 29088
  },
  "seconds": {
    "semi_markov": 5.63534758300375,
    "geometric": 5.784922250000818,
    "direct": 1.7978380000058678
  },
  "training_cases": 897,
  "validation_cases": 234,
  "scores": {
    "semi_markov": {
      "wql9": 0.19132661815170382,
      "mse": 0.4050914261552982,
      "mean_mixture_weight": 0.3185705671706506
    },
    "geometric": {
      "wql9": 0.19572324692368076,
      "mse": 0.41298824502453646,
      "mean_mixture_weight": 0.33864219439749377
    },
    "direct": {
      "wql9": 0.19272542155938793,
      "mse": 0.40600835979532973,
      "mean_mixture_weight": 0.32453091233494813
    },
    "frozen_timesfm3": {
      "wql9": 0.19876065006535057,
      "mse": 0.39498530932677567,
      "mean_mixture_weight": 0.0
    }
  },
  "semi_minus_

## 13. BOOM 专用强基线 Toto 2.0

使用独立 Python 环境和固定 313m 权重 revision，保持同样 234 个案例、原生变量分组和 context≤2048。313m 是当前中等尺寸对照；不能代表 1B/2.5B 或完整官方排行榜。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
checkpoint_path = '/Users/mark/.cache/marktsf-research/huggingface/models--Datadog--Toto-2.0-313m/snapshots/a7bab288f5e95f8606f8306f86659357e1c001ef'
case_scores_path = '/Users/mark/.cache/marktsf-research/r1/factorial_5e3df48b7f0d/case_scores.parquet'
runtime_python = '/Users/mark/.cache/marktsf-research/toto-venv/bin/python'
max_tasks = 48
max_variates = 4
horizon = 32
context_length = 2048
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="matched_toto", repo_root=repo_root, cache_dir=cache_dir, checkpoint_path=checkpoint_path, case_scores_path=case_scores_path, runtime_python=runtime_python, max_tasks=max_tasks, max_variates=max_variates, horizon=horizon, context_length=context_length)

/Users/mark/.cache/marktsf-research/toto-venv/lib/python3.12/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


Toto2 loaded on mps; 108 native groups; patch=32
Toto2 groups 1/108


Toto2 groups 21/108


Toto2 groups 41/108


Toto2 groups 61/108


Toto2 groups 81/108


Toto2 groups 101/108


Toto2 groups 108/108


{
  "stage": "matched_toto",
  "cases": 234,
  "datasets": 18,
  "scores": {
    "wql9": 0.20094767492936363,
    "mse": 0.3922850498620543,
    "mae": 0.15378247362526376,
    "crossing_fraction": 0.0
  },
  "seconds": 15.622072665995802,
  "runtime": {
    "device": "mps",
    "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--Datadog--Toto-2.0-313m/snapshots/a7bab288f5e95f8606f8306f86659357e1c001ef",
    "horizon": 32,
    "decode_block_size": 768,
    "groups": 108,
    "scaler": "official float64 statistics on CPU",
    "torch": "2.14.0",
    "parameters": 312684608
  },
  "cache": "/Users/mark/.cache/marktsf-research/r1/matched_toto_7baad58cc811",
  "sota": false,
  "scope": "313m baseline on fixed development cases; larger Toto2 sizes and complete suite remain"
}


## 14. 固定配置的五随机种子复核

保留数据、训练步数与模型配置，仅改变初始化和抽样 seed。先在每个 dataset 内平均五次优化结果，再对 dataset 做配对 bootstrap，不能把五次 seed 当作五倍独立数据。已有 seed=2021 的结果在检查算法与配置哈希后复用。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/refinement_data_ab1bcdcdbd84'
seeds = [2021, 2022, 2023, 2024, 2025]
steps = 600
batch_size = 64
width = 64
learning_rate = 0.002
age_cap = 128
existing_runs = {'2021': '/Users/mark/.cache/marktsf-research/r1/refinement_fit_f20399c5eca5'}
_ = __import__("runpy").run_path(repo_root + "/src/r1/run.py")["run_step"](step="refinement_seeds", repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, seeds=seeds, steps=steps, batch_size=batch_size, width=width, learning_rate=learning_rate, age_cap=age_cap, existing_runs=existing_runs)

Refiner semi_markov: 5.7s, 28113 parameters


Refiner direct: 1.8s, 29088 parameters


Refiner geometric: 5.8s, 27203 parameters


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/refinement_fit_378feb6ab0c0",
  "device": "mps",
  "steps": 600,
  "seed": 2022,
  "parameters": {
    "semi_markov": 28113,
    "direct": 29088,
    "geometric": 27203
  },
  "seconds": {
    "semi_markov": 5.667684625004767,
    "direct": 1.8122197500051698,
    "geometric": 5.79249858300318
  },
  "training_cases": 897,
  "validation_cases": 234,
  "scores": {
    "semi_markov": {
      "wql9": 0.19506735620061477,
      "mse": 0.4113801864531513,
      "mean_mixture_weight": 0.32580047095904907
    },
    "direct": {
      "wql9": 0.1919769686951729,
      "mse": 0.4014481889733505,
      "mean_mixture_weight": 0.3178544666693207
    },
    "geometric": {
      "wql9": 0.1891051335564614,
      "mse": 0.39526889684267735,
      "mean_mixture_weight": 0.31935567010777094
    },
    "frozen_timesfm3": {
      "wql9": 0.19876065006535057,
      "mse": 0.39498530932677567,
      "mean_mixture_weight": 0.0
    }
  },
  "semi_minus_dir

Refiner semi_markov: 5.6s, 28113 parameters


...

Refiner direct: 1.8s, 29088 parameters


Refiner geometric: 5.8s, 27203 parameters


Refiner semi_markov: 5.7s, 28113 parameters


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/refinement_fit_b09f2b95a239",
  "device": "mps",
  "steps": 600,
  "seed": 2025,
  "parameters": {
    "direct": 29088,
    "geometric": 27203,
    "semi_markov": 28113
  },
  "seconds": {
    "direct": 1.8085072499961825,
    "geometric": 5.809989125002176,
    "semi_markov": 5.669473416994151
  },
  "training_cases": 897,
  "validation_cases": 234,
  "scores": {
    "direct": {
      "wql9": 0.19095926346066572,
      "mse": 0.3985877556299488,
      "mean_mixture_weight": 0.3197511647139446
    },
    "geometric": {
      "wql9": 0.18980767516267708,
      "mse": 0.3970915342132501,
      "mean_mixture_weight": 0.313783156325402
    },
    "semi_markov": {
      "wql9": 0.1922973643141795,
      "mse": 0.40706131147344216,
      "mean_mixture_weight": 0.3223922183748161
    },
    "frozen_timesfm3": {
      "wql9": 0.19876065006535057,
      "mse": 0.39498530932677567,
      "mean_mixture_weight": 0.0
    }
  },
  "semi_minus_dire

{
  "stage": "refinement_seed_replication",
  "seeds": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "datasets": 18,
  "scores": {
    "semi_markov": {
      "wql9": 0.19192554742320084,
      "mse": 0.4052309289448526,
      "mean_mixture_weight": 0.3188611233865179
    },
    "geometric": {
      "wql9": 0.19124350623391573,
      "mse": 0.4005180459889968,
      "mean_mixture_weight": 0.3208746193507086
    },
    "direct": {
      "wql9": 0.192223469515163,
      "mse": 0.4028048847188199,
      "mean_mixture_weight": 0.3185721646259264
    },
    "frozen_timesfm3": {
      "wql9": 0.19876065006535062,
      "mse": 0.39498530932677567,
      "mean_mixture_weight": 0.0
    }
  },
  "semi_minus_direct": {
    "mean": -0.00029792209196217626,
    "ci95": [
      -0.0017870127928744431,
      0.001188238982766194
    ],
    "clusters": 18
  },
  "semi_minus_frozen": {
    "mean": -0.006835102642149766,
    "ci95": [
      -0.024378860179985747,
      0.003331370709145141
  

## 15. 有监督持续时间 × 幅度耦合检验

五 seed 原子混合调整没有证明持续时间的独特收益。此步改用训练前缀内的实际状态转移监督 hazard，随后固定 hazard，再拟合幅度校准。2×2 比较未来 hazard 几何/年龄依赖 × 幅度独立/依赖未来年龄；两个 hazard 臂都保留预测起点的年龄信息。各臂参数分配一致，但这不等于函数容量一致。

以精确的有限状态—年龄递推产生未来占用分布；连续幅度可随未来年龄组做有界位置/尺度变换。混合分布的离散 CRPS 用排序公式计算，训练过程中没有使用未来验证状态。32 个中点支持近似基础分布，另报相同离散化下的冻结模型，避免把数值近似收益当方法收益。

此实现是 neural semi-Markov 强机制对照，不宣称新增理论。固定三个 seed、400 步 hazard + 400 步 emission，不根据验证指标提前停止。主目标仍是后续完整真实套件与强基线，当前仅作开发检验。

结果：年龄耦合 WQL=0.209484，原 TimesFM=0.198761；交互区间跨零，本结构不升级为论文算法。幅度模块是状态条件校准专家，因原支持可能含原子，不能解释为已识别的非原子条件发射。八种仿射混合与单种仿射对照还存在有效容量差异。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/refinement_data_ab1bcdcdbd84'
seeds = [2021, 2022, 2023]
hazard_steps = 400
emission_steps = 400
batch_size = 64
width = 64
learning_rate = 0.002
support_count = 32
age_cap = 128
_ = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('coupling', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, seeds=seeds, hazard_steps=hazard_steps, emission_steps=emission_steps, batch_size=batch_size, width=width, learning_rate=learning_rate, support_count=support_count, age_cap=age_cap)

Coupling age_coupled seed=2021: 1.8s


Coupling age_independent seed=2021: 1.7s


Coupling geometric_coupled seed=2021: 1.7s


Coupling geometric_independent seed=2021: 1.7s


Coupling geometric_independent seed=2022: 1.7s


...

Coupling geometric_independent seed=2023: 1.7s


Coupling geometric_coupled seed=2023: 1.7s


Coupling age_independent seed=2023: 1.7s


Coupling age_coupled seed=2023: 1.6s


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/coupling_ffc57d225a79",
  "validation": {
    "passed": true,
    "checks": [
      "probability conservation",
      "Markov closed form",
      "nonconstant-hazard path enumeration",
      "CRPS versus pairwise definition",
      "finite gradients",
      "ordered quantiles"
    ]
  },
  "stage": "supervised_duration_amplitude",
  "seeds": [
    2021,
    2022,
    2023
  ],
  "device": "mps",
  "training_cases": 897,
  "validation_cases": 234,
  "datasets": 18,
  "support_count": 32,
  "scores": {
    "age_coupled": {
      "wql9": 0.20948406975636724,
      "mse": 0.4011617285930471,
      "gate": 0.3466217600566009
    },
    "age_independent": {
      "wql9": 0.21352171781946275,
      "mse": 0.41552500706821627,
      "gate": 0.34888377068955534
    },
    "geometric_coupled": {
      "wql9": 0.2167198427723289,
      "mse": 0.4539766504130175,
      "gate": 0.34036805318924407
    },
    "geometric_independent": {
      "wql9

## 16. 非平稳价格案例的匹配 TimesFM 强基线

保持第 8 步全部 246 个价格案例、41 个可用资产、H=32 和目标哈希。按市场与起点恢复原生多变量输入，固定 TimesFM 3.0 权重与默认对称平均。价格水平成绩不能单独证明收益信号，下步同时报告累计简单收益误差和经典随机游走。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
checkpoint_path = '/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2'
case_scores_path = '/Users/mark/.cache/marktsf-research/r1/factorial_6acd7984ccf5/case_scores.parquet'
domain = 'price'
max_assets = 24
horizon = 32
context_length = 2048
batch_size = 2
seed = 2021
use_symmetric_averaging = True
_ = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('matched_foundation', repo_root=repo_root, cache_dir=cache_dir, checkpoint_path=checkpoint_path, case_scores_path=case_scores_path, domain=domain, max_assets=max_assets, horizon=horizon, context_length=context_length, batch_size=batch_size, seed=seed, use_symmetric_averaging=use_symmetric_averaging)

/Users/mark/.cache/marktsf-research/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from local directory


TimesFM3 native multivariate inference: 12 groups / 246 cases


{
  "stage": "matched_development_foundation",
  "domain": "price",
  "cases": 246,
  "native_groups": 12,
  "context_max": 2048,
  "horizon": 32,
  "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2",
  "reference_sha256": "7094eac8cee27013b76dc4fb3a7be7c04deb39b5c9ec299ffba21282d20d5076",
  "use_symmetric_averaging": true,
  "prediction_directory": "/Users/mark/.cache/marktsf-research/r1/foundation_predictions_58e9d865b4c1",
  "provenance": {
    "head": "c80482f323e02b8983663c9e1f87a5927698d641",
    "sources": {
      "src/r1/coupling.py": "8f4195a02f99432d1d6adef37e4ef0fd19fdfe0321b9758b76afbd213f5647cb",
      "src/r1/data.py": "8c4cb948f871f2fbcd832cb9bf5c9a240be3cf958b8f9fe37e2d691a1aad7bf7",
      "src/r1/duration.py": "4da1cfa40b29878f8c916253523cac1d4cdded3fd24b3b2b4faac7ce9ba5624c",
      "src/r1/evaluation.py": "1c3c1a71358ddb80190f08f4329cc34084fc2d1bd4d6bc0bb8f182d14e8ac29b

## 17. 金融随机游走、波动率与累计收益对照

仅使用每个起点已知历史。比较点随机游走、算术 Gaussian 随机游走、零漂移对数 Gaussian 随机游走和固定预测期 EWMA 波动率。后两者输出正价格分布，期望价格与中位数不同，统一比较中位数。这里不把 EWMA 称为拟合 GARCH。

在同样案例上重算 TimesFM 和原四臂的价格 WQL/MSE、累计简单收益 WQL/MSE、80% 区间覆盖率及宽度，并报告非法非正价格分位数比例。累计简单收益为 `P_future/P_origin-1`，不等同逐日收益预测；不裁剪基础模型预测。只有六个共同月份，拒绝把重复资产和重叠 origin 当独立证据。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
reference_path = '/Users/mark/.cache/marktsf-research/r1/factorial_6acd7984ccf5/case_scores.parquet'
foundation_predictions = _['prediction_directory']
reference_predictions = '/Users/mark/.cache/marktsf-research/r1/factorial_6acd7984ccf5/predictions'
max_assets = 24
horizon = 32
volatility_window = 96
ewma_alpha = 0.06
_ = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('price_controls', repo_root=repo_root, cache_dir=cache_dir, reference_path=reference_path, foundation_predictions=foundation_predictions, reference_predictions=reference_predictions, max_assets=max_assets, horizon=horizon, volatility_window=volatility_window, ewma_alpha=ewma_alpha)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/financial_controls_d7ef78353253",
  "stage": "matched_financial_controls",
  "cases": 246,
  "assets": 41,
  "calendar_months": 6,
  "volatility_window": 96,
  "ewma_alpha": 0.06,
  "scores": {
    "point_random_walk": {
      "price_wql9": 0.05577732670036767,
      "price_mse": 267.43894198766407,
      "return_wql9": 1.0,
      "return_mse": 0.009422854966054411,
      "coverage80": 0.005335365853658537,
      "interval80_relative_width": 0.0,
      "nonpositive_quantile_fraction": 0.0
    },
    "arithmetic_gaussian_rw": {
      "price_wql9": 0.045353581587029936,
      "price_mse": 267.43894198766407,
      "return_wql9": 0.8738999015203482,
      "return_mse": 0.009422854966054411,
      "coverage80": 0.8454014227642276,
      "interval80_relative_width": 0.21105284639932256,
      "nonpositive_quantile_fraction": 0.0
    },
    "log_gaussian_rw": {
      "price_wql9": 0.04532896467173625,
      "price_mse": 267.43894198766407,

## 18. 已冻结预测的 BOOM 聚合与误差复核

保持原 234 个开发案例不变，重算任务内 sum(pinball)/sum(abs(target))，再按官方低方差名单与 naive MASE=0 规则分组；常规组使用 SeasonalNaive 归一化和 shifted geometric mean。SeasonalNaive 采用官方 notebook 使用的 StatsForecast 模型及 Gaussian 区间，context≤2048；不是无概率区间的同名 GluonTS predictor。实际频率读取 Arrow `freq`，`boom_properties.frequency` 是 Short/Medium/Long 分类，不能传给时间偏移解析器。

所有模型使用同一原始 float64 目标和目标哈希。五 seed/三 seed 在任务内先平均分数，不把它们当独立任务，也不混称预测 ensemble。逐任务、逐案例与原子/非原子损失分解保存在仓库外。

本步仅复核聚合方法；仍不是官方原 horizon、全变量、全窗口和最终测试。上游异常值均值填充不用于本次 WQL 审计，遇到非法分母即报错。

首次复核发现 ds-1613-T 恒定验证段主导相对改善，因此追加全部案例上的 last-value 与历史原子退化分布对照。完整主结果仍保留该任务，同时报告留一任务敏感性，不事后删除任务。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/refinement_data_ab1bcdcdbd84'
foundation_predictions = '/Users/mark/.cache/marktsf-research/r1/foundation_predictions_a71190020118'
toto_predictions = '/Users/mark/.cache/marktsf-research/r1/matched_toto_7baad58cc811/predictions'
refinement_runs = ['/Users/mark/.cache/marktsf-research/r1/refinement_fit_f20399c5eca5', '/Users/mark/.cache/marktsf-research/r1/refinement_fit_378feb6ab0c0', '/Users/mark/.cache/marktsf-research/r1/refinement_fit_e7377308d679', '/Users/mark/.cache/marktsf-research/r1/refinement_fit_be5087f2a174', '/Users/mark/.cache/marktsf-research/r1/refinement_fit_b09f2b95a239']
coupling_run = '/Users/mark/.cache/marktsf-research/r1/coupling_ffc57d225a79'
upstream_leaderboard = '/Users/mark/.cache/marktsf-research/toto/boom/utils/leaderboard.py'
runtime_python = '/Users/mark/.cache/marktsf-research/baseline-venv/bin/python'
horizon = 32
_ = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('aggregation_audit', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, foundation_predictions=foundation_predictions, toto_predictions=toto_predictions, refinement_runs=refinement_runs, coupling_run=coupling_run, upstream_leaderboard=upstream_leaderboard, runtime_python=runtime_python, horizon=horizon)

/Users/mark/.cache/marktsf-research/baseline-venv/lib/python3.12/site-packages/gluonts/time_feature/seasonality.py:47: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  offset = pd.tseries.frequencies.to_offset(freq)
/Users/mark/.cache/marktsf-research/baseline-venv/lib/python3.12/site-packages/gluonts/time_feature/seasonality.py:47: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  offset = pd.tseries.frequencies.to_offset(freq)
/Users/mark/.cache/marktsf-research/baseline-venv/lib/python3.12/site-packages/gluonts/time_feature/seasonality.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset = pd.tseries.frequencies.to_offset(freq)


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/aggregation_audit_7c46a9dc02c0",
  "stage": "official_style_development_aggregation",
  "cases": 234,
  "datasets": 18,
  "scores": {
    "low_variance_unscaled": {
      "atom_direct": {
        "tasks": 3,
        "shifted_geometric_wql9": 1.1703214282775742e-05,
        "task_wql9_mean": 3.064156817903989e-05,
        "historical_case_wql9_mean": 3.064156817903988e-05
      },
      "atom_geometric": {
        "tasks": 3,
        "shifted_geometric_wql9": 1.1700785253225516e-05,
        "task_wql9_mean": 3.0640448725794245e-05,
        "historical_case_wql9_mean": 3.064044872579424e-05
      },
      "atom_semi_markov": {
        "tasks": 3,
        "shifted_geometric_wql9": 1.1700785253225516e-05,
        "task_wql9_mean": 3.0640448725794245e-05,
        "historical_case_wql9_mean": 3.064044872579424e-05
      },
      "coupling_age_coupled": {
        "tasks": 3,
        "shifted_geometric_wql9": 1.159745575495965e-05,
        "

## 19. 经典分布传播能否解决候选“梯度缺口”

有限两状态 semi-Markov 奖励过程：安静态奖励为0，活动态奖励为1或预设的大增量；退出概率随年龄变化。完整整数网格精确递推给出累计分布与期望 CRPS 的 hazard 梯度，并逐场景核对中心有限差分。这里使用 CPU float64 作为数值参考计算，未训练模型。

固定 24 个机制场景 × 32/128 支持预算，比对经典 C51-style categorical projection。支持范围为完整可达奖励界，不人为缩窄；报告边界投影质量、原子质量误差、Wasserstein 距离、梯度误差、符号和资源。最终等权硬分位数的局部零梯度只是诊断对照，不冒称完整 quantile-spline 或可微 OT 实现。

若已有 categorical 方法已经足够，则否定“只需保留 hazard 梯度即可构成创新”的假设；即使发现数值缺口，也必须继续比较 spline/OT 和非均匀网格，且模拟结果不能替代真实 SOTA。

探索性追加：均匀网格在跨度大于最小正奖励时会把正质量投影到0。加入经典非均匀线性投影（0、1及对数间距支持）检查这一误差是否由普通网格设计即可消除。这不是新算法，也不是事先冻结的确认实验。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
horizons = [8, 32]
tails = [8, 32]
tail_probabilities = [0.05, 0.2]
query_probabilities = [0.05, 0.4, 0.7]
supports = [32, 128]
truth_probability = 0.2
finite_difference_step = 0.00001
grid_modes = ['uniform', 'log']
_ = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('propagation_audit', repo_root=repo_root, cache_dir=cache_dir, horizons=horizons, tails=tails, tail_probabilities=tail_probabilities, query_probabilities=query_probabilities, supports=supports, truth_probability=truth_probability, finite_difference_step=finite_difference_step, grid_modes=grid_modes)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/propagation_audit_a081844fe2b5",
  "stage": "classical_distributional_projection_audit",
  "reference_device": "cpu_float64",
  "support_results": {
    "log_32": {
      "scenarios": 24,
      "median_relative_gradient_error": 0.04021373424618688,
      "worst_relative_gradient_error": 0.15017700439556544,
      "same_sign_fraction": 1.0,
      "median_wasserstein1": 1.0231893751043755,
      "worst_atom_mass_error": 0.0,
      "worst_clipped_mass_sum": 4.6495827870860474e-07,
      "median_reference_seconds": 0.00619820799693116,
      "median_categorical_seconds": 0.0030371249995369
    },
    "uniform_32": {
      "scenarios": 24,
      "median_relative_gradient_error": 0.0913352190740372,
      "worst_relative_gradient_error": 0.358905285105162,
      "same_sign_fraction": 1.0,
      "median_wasserstein1": 3.562366424741522,
      "worst_atom_mass_error": 0.6106677984791288,
      "worst_clipped_mass_sum": 2.129170496023264e-08,

## 20. 全变量开发合同与正式预测长度规则

保持原48个开发任务ID，扩展到全部原生变量。short/medium/long的预测长度和窗口公式应用于历史70%前缀；最终官方测试仍隔离。长度不能装入内部验证段的配置明确记录，不按成绩或原子比例过滤。上下文最多2048，同一起点内ffill/bfill，目标不插补。每个源文件、输入与目标记录哈希。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
properties_path = '/Users/mark/.cache/marktsf-research/toto/boom/boom_properties.json'
max_tasks = 48
development_fraction = 0.7
train_fraction = 0.7
context_length = 2048
native_data = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('native_prepare', repo_root=repo_root, cache_dir=cache_dir, properties_path=properties_path, max_tasks=max_tasks, development_fraction=development_fraction, train_fraction=train_fraction, context_length=context_length)

Native contract: ds-1594-T; groups=25


Native contract: ds-2025-H; groups=35


Native contract: ds-883-10S; groups=59


Native contract: ds-231-T; groups=84


Native contract: ds-2175-H; groups=94


...

Native contract: ds-2192-30T; groups=878


Native contract: ds-2133-H; groups=888


Native contract: ds-2433-D; groups=889


Native contract: ds-2400-H; groups=899


{
  "data_dir": "/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d",
  "stage": "native_development_contract",
  "tasks": 48,
  "evaluated_tasks": 48,
  "configurations": 131,
  "groups": 899,
  "series": 965,
  "variate_origins": 17078,
  "scalar_targets": 2613756,
  "horizons": [
    30,
    48,
    60,
    480,
    600,
    720,
    900
  ],
  "skipped": [
    {
      "config": "ds-2192-30T/30T/long",
      "item": "2192",
      "reason": "native_horizon_does_not_fit_development_validation",
      "first_origin": 1435,
      "train_end": 1509
    }
  ],
  "development_fraction": 0.7,
  "train_fraction": 0.7,
  "context_length": 2048,
  "imputation": "ffill then bfill using origin-available context; all-missing context set to zero; targets not imputed",
  "groups_sha": "d69f01e140a299c1403edaeb7aad18abb7aab8d604de4784198e6db9698ecd62",
  "sota": false,
  "properties_sha": "eddcd09ba5d66c5a8d6d0b3b294dde001975cbf53a063b51d94ac0b5c6ed8383",
  "scope": "Official horizon/wi

## 21. 全变量正式长度开发基线：timesfm3

899个原生变量组全部执行；固定权重、数据哈希与环境版本。TimesFM保持默认对称平均；Toto在MPS使用官方CPU float64统计量；SeasonalNaive使用StatsForecast概率区间。中断后仅复用身份完全匹配的输出，不替换失败模型。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d'
model_name = 'timesfm3'
checkpoint = '/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2'
runtime_python = '/Users/mark/.cache/marktsf-research/venv/bin/python'
batch_size = 2
native_timesfm = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('native_forecast', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, model_name=model_name, checkpoint=checkpoint, runtime_python=runtime_python, batch_size=batch_size)

Loading weights from local directory
timesfm3: 1/899 groups, H=48, V=19


timesfm3: 11/899 groups, H=48, V=19


timesfm3: 21/899 groups, H=480, V=19


timesfm3: 31/899 groups, H=48, V=34


timesfm3: 41/899 groups, H=60, V=1


...

timesfm3: 871/899 groups, H=720, V=1


timesfm3: 881/899 groups, H=48, V=2


timesfm3: 891/899 groups, H=48, V=1


timesfm3: 899/899 groups, H=720, V=1


{
  "model": "timesfm3",
  "data_dir": "/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d",
  "predictions": "/Users/mark/.cache/marktsf-research/r1/native_timesfm3_8db5c4c69fad/predictions",
  "groups": 899,
  "variate_origins": 17078,
  "runtime": {
    "model": "timesfm3",
    "device": "mps",
    "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--google--timesfm-3.0-pytorch/snapshots/43046b85ec22d584a13f8098c2ed39c889e129c2",
    "groups": 899,
    "seconds": 209.8509544999979,
    "symmetric_averaging": true,
    "timing_scope": "processing loop, excludes model load; resumed outputs skipped",
    "scaler": "native"
  },
  "sota": false
}


## 22. 全变量正式长度开发基线：toto2_313m

899个原生变量组全部执行；固定权重、数据哈希与环境版本。TimesFM保持默认对称平均；Toto在MPS使用官方CPU float64统计量；SeasonalNaive使用StatsForecast概率区间。中断后仅复用身份完全匹配的输出，不替换失败模型。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d'
model_name = 'toto2_313m'
checkpoint = '/Users/mark/.cache/marktsf-research/huggingface/models--Datadog--Toto-2.0-313m/snapshots/a7bab288f5e95f8606f8306f86659357e1c001ef'
runtime_python = '/Users/mark/.cache/marktsf-research/toto-venv/bin/python'
batch_size = 2
native_toto = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('native_forecast', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, model_name=model_name, checkpoint=checkpoint, runtime_python=runtime_python, batch_size=batch_size)

/Users/mark/.cache/marktsf-research/toto-venv/lib/python3.12/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


toto2_313m: 1/899 groups, H=48, V=19


toto2_313m: 11/899 groups, H=48, V=19


toto2_313m: 21/899 groups, H=480, V=19


toto2_313m: 31/899 groups, H=48, V=34


...

toto2_313m: 871/899 groups, H=720, V=1


toto2_313m: 881/899 groups, H=48, V=2


toto2_313m: 891/899 groups, H=48, V=1


toto2_313m: 899/899 groups, H=720, V=1


{
  "model": "toto2_313m",
  "data_dir": "/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d",
  "predictions": "/Users/mark/.cache/marktsf-research/r1/native_toto2_313m_48270236c70a/predictions",
  "groups": 899,
  "variate_origins": 17078,
  "runtime": {
    "model": "toto2_313m",
    "device": "mps",
    "checkpoint": "/Users/mark/.cache/marktsf-research/huggingface/models--Datadog--Toto-2.0-313m/snapshots/a7bab288f5e95f8606f8306f86659357e1c001ef",
    "groups": 899,
    "seconds": 97.22134704099881,
    "symmetric_averaging": false,
    "timing_scope": "processing loop, excludes model load; resumed outputs skipped",
    "scaler": "official CPU float64 statistics"
  },
  "sota": false
}


## 23. 全变量正式长度开发基线：seasonalnaive

899个原生变量组全部执行；固定权重、数据哈希与环境版本。TimesFM保持默认对称平均；Toto在MPS使用官方CPU float64统计量；SeasonalNaive使用StatsForecast概率区间。中断后仅复用身份完全匹配的输出，不替换失败模型。

In [3]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d'
model_name = 'seasonalnaive'
checkpoint = None
runtime_python = '/Users/mark/.cache/marktsf-research/baseline-venv/bin/python'
batch_size = 2
native_naive = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('native_forecast', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, model_name=model_name, checkpoint=checkpoint, runtime_python=runtime_python, batch_size=batch_size)

/Users/mark/.cache/marktsf-research/baseline-venv/lib/python3.12/site-packages/gluonts/time_feature/seasonality.py:47: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  offset = pd.tseries.frequencies.to_offset(freq)
/Users/mark/.cache/marktsf-research/baseline-venv/lib/python3.12/site-packages/gluonts/time_feature/seasonality.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset = pd.tseries.frequencies.to_offset(freq)


seasonalnaive: 1/899 groups, H=48, V=19
seasonalnaive: 11/899 groups, H=48, V=19
seasonalnaive: 21/899 groups, H=480, V=19


seasonalnaive: 31/899 groups, H=48, V=34
seasonalnaive: 41/899 groups, H=60, V=1
seasonalnaive: 51/899 groups, H=60, V=1
seasonalnaive: 61/899 groups, H=48, V=4
seasonalnaive: 71/899 groups, H=48, V=4


/Users/mark/.cache/marktsf-research/baseline-venv/lib/python3.12/site-packages/gluonts/time_feature/seasonality.py:47: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  offset = pd.tseries.frequencies.to_offset(freq)


seasonalnaive: 81/899 groups, H=480, V=4
seasonalnaive: 91/899 groups, H=48, V=1
seasonalnaive: 101/899 groups, H=48, V=3
seasonalnaive: 111/899 groups, H=48, V=3
seasonalnaive: 121/899 groups, H=60, V=3
seasonalnaive: 131/899 groups, H=60, V=3
seasonalnaive: 141/899 groups, H=600, V=3


...

seasonalnaive: 711/899 groups, H=480, V=92


seasonalnaive: 721/899 groups, H=48, V=1
seasonalnaive: 731/899 groups, H=48, V=64


seasonalnaive: 741/899 groups, H=48, V=64
seasonalnaive: 751/899 groups, H=48, V=1
seasonalnaive: 761/899 groups, H=48, V=1
seasonalnaive: 771/899 groups, H=480, V=1
seasonalnaive: 781/899 groups, H=48, V=1


seasonalnaive: 791/899 groups, H=48, V=1
seasonalnaive: 801/899 groups, H=48, V=1
seasonalnaive: 811/899 groups, H=48, V=1
seasonalnaive: 821/899 groups, H=480, V=1
seasonalnaive: 831/899 groups, H=60, V=1
seasonalnaive: 841/899 groups, H=60, V=1
seasonalnaive: 851/899 groups, H=48, V=1
seasonalnaive: 861/899 groups, H=48, V=1
seasonalnaive: 871/899 groups, H=720, V=1
seasonalnaive: 881/899 groups, H=48, V=2
seasonalnaive: 891/899 groups, H=48, V=1
seasonalnaive: 899/899 groups, H=720, V=1


{
  "model": "seasonalnaive",
  "data_dir": "/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d",
  "predictions": "/Users/mark/.cache/marktsf-research/r1/native_seasonalnaive_559a3f844dec/predictions",
  "groups": 899,
  "variate_origins": 17078,
  "runtime": {
    "model": "seasonalnaive",
    "device": "cpu",
    "checkpoint": null,
    "groups": 899,
    "seconds": 8.796541333002097,
    "symmetric_averaging": false,
    "timing_scope": "processing loop, excludes model load; resumed outputs skipped",
    "scaler": "native"
  },
  "sota": false
}


## 24. 全变量开发比较与误差切片

任务内累加WQL分子/分母；按固定低方差名单与实际naive MASE分组。无效分数保留并中止对应完整聚合，不填均值、不默默删除配置。目标恒定性只在评分后分解，不用于选择模型或任务。此处只有强基线，无新算法或SOTA声明。

In [4]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d'
model_directories = {'timesfm3': native_timesfm['predictions'], 'toto2_313m': native_toto['predictions'], 'seasonalnaive': native_naive['predictions']}
upstream_leaderboard = '/Users/mark/.cache/marktsf-research/toto/boom/utils/leaderboard.py'
native_comparison = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('native_compare', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, model_directories=model_directories, upstream_leaderboard=upstream_leaderboard)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/native_comparison_0f7cddd7a3cb",
  "stage": "native_development_comparison",
  "tasks": 48,
  "configurations": 131,
  "variate_origins": 17078,
  "scores": {
    "low_variance_unscaled": {
      "last_value": {
        "configurations": 12,
        "shifted_geometric_wql9": -3.3881317890172014e-21,
        "task_wql9_mean": 0.0,
        "invalid_configurations": 0
      },
      "seasonalnaive": {
        "configurations": 12,
        "shifted_geometric_wql9": 0.00027819866534417424,
        "task_wql9_mean": 1.918900251487497,
        "invalid_configurations": 0
      },
      "timesfm3": {
        "configurations": 12,
        "shifted_geometric_wql9": 0.000143386990664655,
        "task_wql9_mean": 1.2818582108298124,
        "invalid_configurations": 0
      },
      "toto2_313m": {
        "configurations": 12,
        "shifted_geometric_wql9": 1.7498262646559647e-05,
        "task_wql9_mean": 0.0001420736232904916,
        "in

## 25. 全部开发配置的事件路径误差归因

按历史是否识别到原子、起点状态及未来是否转变，形成互斥且完备的五类事件。未来状态只用于事后诊断，不能进入预测输入或筛选benchmark。各类贡献严格相加为全部131配置未缩放算术WQL；这不是官方几何主分数。年龄切片混合不同任务和horizon，仅作为新假设线索，不能推断因果或独立持续时间收益。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d'
comparison_dir = '/Users/mark/.cache/marktsf-research/r1/native_comparison_0f7cddd7a3cb'
native_diagnostics = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('native_diagnose', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, comparison_dir=comparison_dir)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/native_diagnostics_049526945177",
  "stage": "native_error_attribution",
  "configurations": 131,
  "variate_origins": 17078,
  "events": [
    {
      "model": "last_value",
      "event": "active_enters_atom",
      "macro_wql_contribution": 0.11777325338067748,
      "variate_origins": 972,
      "datasets": 17,
      "points": 158418,
      "atom_points": 54686,
      "macro_error_share": 0.10034510165505739
    },
    {
      "model": "last_value",
      "event": "active_stays_active",
      "macro_wql_contribution": 0.11996176402582033,
      "variate_origins": 3047,
      "datasets": 22,
      "points": 350712,
      "atom_points": 0,
      "macro_error_share": 0.10220975527424717
    },
    {
      "model": "last_value",
      "event": "atom_exit",
      "macro_wql_contribution": 0.07973680444471065,
      "variate_origins": 1717,
      "datasets": 17,
      "points": 469962,
      "atom_points": 313609,
      "macro_error_sh

## 26. boom 连续事件持续性表示检验

固定三臂：近期上下文、上下文+真实年龄、上下文+训练期状态/任务匹配的随机年龄。固定 seeds=2021–2023，HGB每个臂80步，无early stopping；H=32的事件Brier及正尺度累计增量变换pinball为主要机制终点，H=1/8仅作敏感性检查。

状态由当前增量与t-1的因果EWMA尺度比较，预测累计增量的尺度则冻结于可见的t，并乘sqrt(H)。尺度下限为0.001倍历史增量RMS，保留单位等变性；完全恒定历史尺度为0，这些案例仍进入事件/原始值评分，累计增量预测退化为0，未定义的变换目标不进入辅助回归。所有模型统一对分位数排序，记录排序前交叉率。

训练最高标签索引严格早于内部训练边界，三个horizon共用max(H) purge。金融跨市场使用共同日历截止日，不能对每只股票独立重切时间；不使用单月IID bootstrap。此为经典HGB的表示效用探针，不是新算法、联合路径识别或完整BOOM SOTA评测。

首次执行在拟合前遇到pandas 3只读数组的原位权重缩放错误；已改为显式新数组，模型/数据/预算未改变。以下保存修正后的实际运行。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'boom'
horizons = [1, 8, 32]
max_tasks = 48
max_assets = 24
train_origins = 64
validation_origins = 32
alpha = 0.06
threshold = 1.5
floor_ratio = 0.001
seeds = [2021, 2022, 2023]
max_iter = 80
threads = 4
continuous_boom = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('continuous_probe', repo_root=repo_root, cache_dir=cache_dir, domain=domain, horizons=horizons, max_tasks=max_tasks, max_assets=max_assets, train_origins=train_origins, validation_origins=validation_origins, alpha=alpha, threshold=threshold, floor_ratio=floor_ratio, seeds=seeds, max_iter=max_iter, threads=threads)

Continuous boom: 250818 rows; train=164160; validation=86658; tasks=47; series=964


Continuous boom: random_age seed=2021, quantile=0.1


Continuous boom: random_age seed=2021, quantile=0.5


Continuous boom: random_age seed=2021, quantile=0.9


Continuous boom: random_age seed=2021 completed in 50.2s


...

Continuous boom: context seed=2023, quantile=0.1


Continuous boom: context seed=2023, quantile=0.5


Continuous boom: context seed=2023, quantile=0.9


Continuous boom: context seed=2023 completed in 76.2s


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/continuous_probe_boom_5e61efe61ce8",
  "data": {
    "sources": [
      {
        "dataset": "ds-1594-T",
        "total_variates": 19,
        "selected_variates": 19,
        "hashes": {
          "input/BOOM/ds-1594-T/data-00000-of-00001.arrow": "543132874147c3f64bd9d3e14526757c7bf243068ebe200433fe7231da8dd410"
        }
      },
      {
        "dataset": "ds-2025-H",
        "total_variates": 34,
        "selected_variates": 34,
        "hashes": {
          "input/BOOM/ds-2025-H/data-00000-of-00001.arrow": "88cc3e38f515445b269e82c79a9bc583794d528f22a3f65465bccc718c0816c0"
        }
      },
      {
        "dataset": "ds-883-10S",
        "total_variates": 1,
        "selected_variates": 1,
        "hashes": {
          "input/BOOM/ds-883-10S/data-00000-of-00001.arrow": "9f5f3bf07bd00b81313ffdfa001a213818d842fc20ba5e8b56f41a1f69a62c5f"
        }
      },
      {
        "dataset": "ds-231-T",
        "total_variates": 4,
      

## 27. price 连续事件持续性表示检验

固定三臂：近期上下文、上下文+真实年龄、上下文+训练期状态/任务匹配的随机年龄。固定 seeds=2021–2023，HGB每个臂80步，无early stopping；H=32的事件Brier及正尺度累计增量变换pinball为主要机制终点，H=1/8仅作敏感性检查。

状态由当前增量与t-1的因果EWMA尺度比较，预测累计增量的尺度则冻结于可见的t，并乘sqrt(H)。尺度下限为0.001倍历史增量RMS，保留单位等变性；完全恒定历史尺度为0，这些案例仍进入事件/原始值评分，累计增量预测退化为0，未定义的变换目标不进入辅助回归。所有模型统一对分位数排序，记录排序前交叉率。

训练最高标签索引严格早于内部训练边界，三个horizon共用max(H) purge。金融跨市场使用共同日历截止日，不能对每只股票独立重切时间；不使用单月IID bootstrap。此为经典HGB的表示效用探针，不是新算法、联合路径识别或完整BOOM SOTA评测。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
domain = 'price'
horizons = [1, 8, 32]
max_tasks = 48
max_assets = 24
train_origins = 64
validation_origins = 32
alpha = 0.06
threshold = 1.5
floor_ratio = 0.001
seeds = [2021, 2022, 2023]
max_iter = 80
threads = 4
continuous_price = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('continuous_probe', repo_root=repo_root, cache_dir=cache_dir, domain=domain, horizons=horizons, max_tasks=max_tasks, max_assets=max_assets, train_origins=train_origins, validation_origins=validation_origins, alpha=alpha, threshold=threshold, floor_ratio=floor_ratio, seeds=seeds, max_iter=max_iter, threads=threads)

Continuous price: 11808 rows; train=7872; validation=3936; tasks=2; series=41


Continuous price: random_age seed=2021, quantile=0.1


Continuous price: random_age seed=2021, quantile=0.5


Continuous price: random_age seed=2021, quantile=0.9


Continuous price: random_age seed=2021 completed in 10.3s


...

Continuous price: context seed=2023, quantile=0.1


Continuous price: context seed=2023, quantile=0.5


Continuous price: context seed=2023, quantile=0.9


Continuous price: context seed=2023 completed in 9.8s


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/continuous_probe_price_e14859130c64",
  "data": {
    "sources": [
      {
        "dataset": "SP500",
        "sha256": "75bbb7f50e3be5d29aea1b40d0417e2796ceef3d8a63ef7937c9923884a41e5d",
        "columns": [
          "ODFL",
          "BNY",
          "VRTX",
          "APA",
          "WST",
          "RJF",
          "AEP",
          "CNC",
          "PSX",
          "LYV",
          "BRO",
          "SW",
          "FTV",
          "CHRW",
          "SYY",
          "HAS",
          "EXPD",
          "FRT",
          "DRI",
          "PGR",
          "WAB",
          "ISRG",
          "SATS",
          "TSN"
        ],
        "development_rows": 1056,
        "train_end": 739,
        "development_last_date": "2022-03-11"
      },
      {
        "dataset": "CSI500",
        "sha256": "bedd0a2210a1de4b1e8f6bf997eaf060357b1ee915fc97a972693cd25dbd6731",
        "columns": [
          "002080",
          "002926",
          "6007

## 28. BOOM 年龄探针的尺度、任务与随机游走复核

主要终点保持不变。按尺度下限触发与训练任务覆盖分解H32效应，贡献之和必须还原主比较；验证完全相同预测是否被错误当作独立seed证据。补齐起点冻结尺度的Gaussian随机游走，对照原始端点WQL及金融累计收益误差。此处没有重新选模型或删除不利任务。

In [1]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
run_dir = '/Users/mark/.cache/marktsf-research/r1/continuous_probe_boom_5e61efe61ce8'
audit_28 = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('continuous_audit', repo_root=repo_root, cache_dir=cache_dir, run_dir=run_dir)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/continuous_audit_a473e3cf5cfd",
  "stage": "continuous_probe_attribution",
  "domain": "boom",
  "validation_only_tasks": [
    "ds-1621-D",
    "ds-2433-D",
    "ds-2537-D"
  ],
  "floor_active_fraction": 0.16104687391816105,
  "zero_scale_cases": 612,
  "h32_effects": [
    {
      "horizon": 32,
      "metric": "brier",
      "floor_active": 0.0,
      "training_seen": false,
      "cases": 1526,
      "tasks": 3,
      "age_minus_context_contribution": -9.541562558233205e-05,
      "age_minus_random_contribution": -4.393000982246745e-05
    },
    {
      "horizon": 32,
      "metric": "brier",
      "floor_active": 0.0,
      "training_seen": true,
      "cases": 22708,
      "tasks": 41,
      "age_minus_context_contribution": 9.753575488784301e-05,
      "age_minus_random_contribution": 0.00027645163978712566
    },
    {
      "horizon": 32,
      "metric": "brier",
      "floor_active": 1.0,
      "training_seen": true,
    

## 29. 金融 年龄探针的尺度、任务与随机游走复核

主要终点保持不变。按尺度下限触发与训练任务覆盖分解H32效应，贡献之和必须还原主比较；验证完全相同预测是否被错误当作独立seed证据。补齐起点冻结尺度的Gaussian随机游走，对照原始端点WQL及金融累计收益误差。此处没有重新选模型或删除不利任务。

In [2]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
run_dir = '/Users/mark/.cache/marktsf-research/r1/continuous_probe_price_e14859130c64'
audit_29 = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('continuous_audit', repo_root=repo_root, cache_dir=cache_dir, run_dir=run_dir)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/continuous_audit_ab473e07a85a",
  "stage": "continuous_probe_attribution",
  "domain": "price",
  "validation_only_tasks": [],
  "floor_active_fraction": 0.0,
  "zero_scale_cases": 0,
  "h32_effects": [
    {
      "horizon": 32,
      "metric": "brier",
      "floor_active": 0.0,
      "training_seen": true,
      "cases": 1312,
      "tasks": 2,
      "age_minus_context_contribution": -0.001003113160721642,
      "age_minus_random_contribution": -0.0009644137831387251
    },
    {
      "horizon": 32,
      "metric": "transformed_pinball9",
      "floor_active": 0.0,
      "training_seen": true,
      "cases": 1312,
      "tasks": 2,
      "age_minus_context_contribution": 0.00021135636774919847,
      "age_minus_random_contribution": -2.6183707370202042e-05
    }
  ],
  "raw_endpoint_scores": [
    {
      "horizon": 1,
      "arm": "context",
      "tasks": 2,
      "endpoint_wql9": 0.013553233867426101,
      "undefined_tasks": 

## 30. boom 原生多通道共同事件年龄的四臂检验

复用第26–27步全部案例、切分、H=1/8/32标签和尺度。BOOM按原始Arrow行恢复通道组；金融固定原有效资产池，只使用同市场、同日收盘后可见数据，不跨市场对齐。整段历史逐时排除目标，再计算peer状态比例（固定阈值0.25）、共同状态与年龄。零peer案例保留为零特征并标记；单peer不能解释为群体冲击。原开发前缀缺失值排除规则继承，不能称为新的前瞻队列。

四臂：A自身上下文及自身年龄；B增加普通peer增量/事件率、lag/rolling、当前共同状态和左删失信息；C仅在B上增加真实共同状态log年龄；D同维度，年龄从训练期原生组×共同状态×删失状态池抽取，记录fallback。A也控制peer数量/可用性和历史长度。A/B/C仅拟合1次，D运行3个placebo种子，先平均种子再按观测簇比较。

主要门槛仍为H32事件Brier及累计增量变换pinball：C须同时优于B和D，且原始尺度结果不能弱于A。H1/8只作敏感性检查，零尺度/尺度下限均保留。案例曾用于候选选择，因此即使通过也仅是开发机制线索，不是独立确认、新算法或SOTA。


In [30]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
base_dir = '/Users/mark/.cache/marktsf-research/r1/continuous_probe_boom_5e61efe61ce8'
event_fraction = 0.25
seeds = [2021, 2022, 2023]
max_iter = 80
threads = 4
peer_boom = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('peer_probe', repo_root=repo_root, cache_dir=cache_dir, base_dir=base_dir, event_fraction=event_fraction, seeds=seeds, max_iter=max_iter, threads=threads)

Peer boom: 250818 fixed cases, 47 native groups; cache=/Users/mark/.cache/marktsf-research/r1/peer_probe_boom_3417d074185e


Continuous boom: random_age seed=2021, quantile=0.1


Continuous boom: random_age seed=2021, quantile=0.5


Continuous boom: random_age seed=2021, quantile=0.9


Continuous boom: random_age seed=2021 completed in 142.9s


...

Continuous boom: random_age seed=2023, quantile=0.1


Continuous boom: random_age seed=2023, quantile=0.5


Continuous boom: random_age seed=2023, quantile=0.9


Continuous boom: random_age seed=2023 completed in 145.9s


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/peer_probe_boom_3417d074185e",
  "data": {
    "domain": "boom",
    "base_dir": "/Users/mark/.cache/marktsf-research/r1/continuous_probe_boom_5e61efe61ce8",
    "base_cases_sha": "9be5b6aaa8af2099fc4b06372571d9eabbfe65110cf83bf18693f557df85ce1b",
    "base_contract_sha": "713bba764eb0e392d2d894cd03fcaff332fd399841fc710fb48433a10f25136c",
    "validation_peer_coverage": [
      {
        "peer_count": 0.0,
        "cases": 2058,
        "tasks": 22
      },
      {
        "peer_count": 1.0,
        "cases": 192,
        "tasks": 1
      },
      {
        "peer_count": 2.0,
        "cases": 1152,
        "tasks": 4
      },
      {
        "peer_count": 3.0,
        "cases": 768,
        "tasks": 2
      },
      {
        "peer_count": 4.0,
        "cases": 480,
        "tasks": 1
      },
      {
        "peer_count": 7.0,
        "cases": 336,
        "tasks": 1
      },
      {
        "peer_count": 8.0,
        "cases": 864,
  

## 31. price 原生多通道共同事件年龄的四臂检验

复用第26–27步全部案例、切分、H=1/8/32标签和尺度。BOOM按原始Arrow行恢复通道组；金融固定原有效资产池，只使用同市场、同日收盘后可见数据，不跨市场对齐。整段历史逐时排除目标，再计算peer状态比例（固定阈值0.25）、共同状态与年龄。零peer案例保留为零特征并标记；单peer不能解释为群体冲击。原开发前缀缺失值排除规则继承，不能称为新的前瞻队列。

四臂：A自身上下文及自身年龄；B增加普通peer增量/事件率、lag/rolling、当前共同状态和左删失信息；C仅在B上增加真实共同状态log年龄；D同维度，年龄从训练期原生组×共同状态×删失状态池抽取，记录fallback。A也控制peer数量/可用性和历史长度。A/B/C仅拟合1次，D运行3个placebo种子，先平均种子再按观测簇比较。

主要门槛仍为H32事件Brier及累计增量变换pinball：C须同时优于B和D，且原始尺度结果不能弱于A。H1/8只作敏感性检查，零尺度/尺度下限均保留。案例曾用于候选选择，因此即使通过也仅是开发机制线索，不是独立确认、新算法或SOTA。


In [31]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
base_dir = '/Users/mark/.cache/marktsf-research/r1/continuous_probe_price_e14859130c64'
event_fraction = 0.25
seeds = [2021, 2022, 2023]
max_iter = 80
threads = 4
peer_price = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('peer_probe', repo_root=repo_root, cache_dir=cache_dir, base_dir=base_dir, event_fraction=event_fraction, seeds=seeds, max_iter=max_iter, threads=threads)

Peer price: 11808 fixed cases, 2 native groups; cache=/Users/mark/.cache/marktsf-research/r1/peer_probe_price_bda6225f5e88


Continuous price: random_age seed=2021, quantile=0.1


Continuous price: random_age seed=2021, quantile=0.5


Continuous price: random_age seed=2021, quantile=0.9


Continuous price: random_age seed=2021 completed in 39.3s


...

Continuous price: random_age seed=2023, quantile=0.1


Continuous price: random_age seed=2023, quantile=0.5


Continuous price: random_age seed=2023, quantile=0.9


Continuous price: random_age seed=2023 completed in 14.8s


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/peer_probe_price_bda6225f5e88",
  "data": {
    "domain": "price",
    "base_dir": "/Users/mark/.cache/marktsf-research/r1/continuous_probe_price_e14859130c64",
    "base_cases_sha": "9368ea5b84b3471935cae099332c45123637d946760109000200cb33d6fa121d",
    "base_contract_sha": "a744d3c76a7a911af464ac8c66ad1dccb780aec8f8fe30c7817b938e2b27e874",
    "validation_peer_coverage": [
      {
        "peer_count": 16.0,
        "cases": 1632,
        "tasks": 1
      },
      {
        "peer_count": 23.0,
        "cases": 2304,
        "tasks": 1
      }
    ],
    "target_cases_unchanged": true,
    "source_values_and_features_exactly_reproduced": true,
    "calendar_cutoff": "2020-12-08 00:00:00",
    "inherited_exclusions": [
      {
        "dataset": "CSI500",
        "item": "002926",
        "reason": "nonfinite_or_nonpositive_price"
      },
      {
        "dataset": "CSI500",
        "item": "688297",
        "reason": "nonfinite_or_

## 32. BOOM 四臂预测、peer数量与尺度归因复核

逐臂核验全部保存预测键、四臂特征集合和原始Brier/pinball分数，金融另核验累计收益误差。先逐案例平均随机seed，然后按原任务或共同月份权重分解无peer、单peer、多peer、尺度下限和训练组覆盖切片；切片之和必须还原H32主要效应。补齐Gaussian随机游走及H32原始尺度guardrail，禁止用混合horizon均值代替。

本轮变换空间分位数未单独保存；变换pinball只检查案例完整性与聚合守恒，不宣称它已从独立预测记录复算。均值门槛并非显著性或SOTA证明；无peer案例仍可能因共享模型拟合改变，不能假设其差值必为零。


In [32]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
run_dir = '/Users/mark/.cache/marktsf-research/r1/peer_probe_boom_3417d074185e'
peer_audit_32 = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('peer_audit', repo_root=repo_root, cache_dir=cache_dir, run_dir=run_dir)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/peer_audit_4e2da6478c35",
  "stage": "peer_age_attribution_audit",
  "domain": "boom",
  "prediction_checks": [
    {
      "arm": "context",
      "seed": 2021,
      "cases": 86658,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "own_context",
      "seed": 2021,
      "cases": 86658,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "random_age",
      "seed": 2021,
      "cases": 86658,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "random_age",
      "seed": 2022,
      "cases": 86658,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "random_age",
      "seed": 2023,
      "cases": 86658,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
  

## 33. 金融 四臂预测、peer数量与尺度归因复核

逐臂核验全部保存预测键、四臂特征集合和原始Brier/pinball分数，金融另核验累计收益误差。先逐案例平均随机seed，然后按原任务或共同月份权重分解无peer、单peer、多peer、尺度下限和训练组覆盖切片；切片之和必须还原H32主要效应。补齐Gaussian随机游走及H32原始尺度guardrail，禁止用混合horizon均值代替。

本轮变换空间分位数未单独保存；变换pinball只检查案例完整性与聚合守恒，不宣称它已从独立预测记录复算。均值门槛并非显著性或SOTA证明；无peer案例仍可能因共享模型拟合改变，不能假设其差值必为零。


In [33]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
run_dir = '/Users/mark/.cache/marktsf-research/r1/peer_probe_price_bda6225f5e88'
peer_audit_33 = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('peer_audit', repo_root=repo_root, cache_dir=cache_dir, run_dir=run_dir)

{
  "cache": "/Users/mark/.cache/marktsf-research/r1/peer_audit_a0e48ac76fbf",
  "stage": "peer_age_attribution_audit",
  "domain": "price",
  "prediction_checks": [
    {
      "arm": "context",
      "seed": 2021,
      "cases": 3936,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "own_context",
      "seed": 2021,
      "cases": 3936,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "random_age",
      "seed": 2021,
      "cases": 3936,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "random_age",
      "seed": 2022,
      "cases": 3936,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {
      "arm": "random_age",
      "seed": 2023,
      "cases": 3936,
      "raw_and_event_scores_match": true,
      "zero_scale_forecast_verified": true
    },
    {


## 34. 原生Toto的因果校准与事后时段误差空间

固定全部131配置、899原生组和既有Toto预测。仿射分位数族为median + bias×半80%区间宽度 + spread×原分位数偏差，35点固定grid包含identity，零宽度不人为加噪声。它是经典校准对照，不是R1新方法。

可执行三臂：同配置池、同变量扩展历史、同变量最近3条完整路径。只有旧origin+H≤当前origin的路径标签可用于选参数；至少2条完整旧窗口才启用，否则identity。当前目标及事后事件划分均不能参与选择。滚动校准属于adapted协议，不能与原zero-shot基线混称同一训练设定。

事后oracle四臂：每条路径最优参数、前后半段分别最优、按首个真实原子状态转换分段、任意最佳分割点。它们读取当前未来标签，只衡量这个仿射族的经验误差空间，不能作为预测方法、可达性能上限或SOTA成绩。无内部事件边界时，事件oracle回退固定半段；边界与最佳cut不能进入可执行臂。

主评分保留既有原生完整路径WQL及低方差划分；先精确复现原Toto配置分数。事后事件贡献仅分解算术配置WQL，不冒充几何榜单分解。该步骤用于判断持续时间机制是否仍有超出普通幅度/位置校准的具体缺口。

In [34]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
data_dir = '/Users/mark/.cache/marktsf-research/r1/native_data_1aa1722b8d0d'
prediction_dir = '/Users/mark/.cache/marktsf-research/r1/native_toto2_313m_48270236c70a/predictions'
comparison_dir = '/Users/mark/.cache/marktsf-research/r1/native_comparison_0f7cddd7a3cb'
biases = [-0.5, -0.25, 0.0, 0.25, 0.5]
spreads = [0.0, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
minimum_windows = 2
recent_windows = 3
native_calibration = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('calibration_probe', repo_root=repo_root, cache_dir=cache_dir, data_dir=data_dir, prediction_dir=prediction_dir, comparison_dir=comparison_dir, biases=biases, spreads=spreads, minimum_windows=minimum_windows, recent_windows=recent_windows)

Native calibration: 100/899 groups


Native calibration: 200/899 groups


Native calibration: 300/899 groups


Native calibration: 400/899 groups


Native calibration: 500/899 groups


Native calibration: 600/899 groups


Native calibration: 700/899 groups


Native calibration: 800/899 groups


Native calibration: 899/899 groups


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/native_calibration_77bd52389598",
  "validation": {
    "pending_future_labels_cannot_select_parameters": true,
    "labels_at_current_origin_are_available": true,
    "cold_start_identity": true,
    "all_grid_quantiles_monotone": true,
    "identity_tie_priority": true
  },
  "stage": "native_causal_calibration_and_hindsight_diagnostic",
  "groups": 899,
  "variate_origins": 17078,
  "grid": [
    [
      0.0,
      1.0
    ],
    [
      0.0,
      0.75
    ],
    [
      0.0,
      1.25
    ],
    [
      -0.25,
      1.0
    ],
    [
      0.25,
      1.0
    ],
    [
      0.0,
      0.5
    ],
    [
      0.0,
      1.5
    ],
    [
      -0.25,
      0.75
    ],
    [
      -0.25,
      1.25
    ],
    [
      0.25,
      0.75
    ],
    [
      0.25,
      1.25
    ],
    [
      -0.5,
      1.0
    ],
    [
      0.5,
      1.0
    ],
    [
      -0.25,
      0.5
    ],
    [
      -0.25,
      1.5
    ],
    [
      0.25,


## 35. 已知正确预测分布的事后择优负对照

第34步因果校准均退化，但oracle明显改善。为避免把事后择优收益误判为可学习误差，固定6个已知分布设置：H=48/480的Gaussian AR(1)（rho=0/0.9，x0=0）及正价格对数随机游走（sigma=0.02，保证期望价格为100的已知drift）。直接使用解析条件分位数，基线在每个horizon都是正确的Bayes分位数。

每个设置生成2048组相互独立的A/B未来路径，路径内部允许时间相关性。同第34步35点grid，oracle在A上选整段或固定半段参数，再把这些参数原样用于独立B路径。比较A上事后改善和B上迁移误差；统计单位是独立路径对，绝非单个时间点。该负对照不拟合Toto、不证明Toto已校准，也不代表真实BOOM分布；它只检验oracle改善是否足以证明存在可学习信号。

In [35]:
repo_root = '/Users/mark/Git/marktsf'
cache_dir = '/Users/mark/.cache/marktsf-research/r1'
horizons = [48, 480]
replicates = 2048
rhos = [0.0, 0.9]
seed = 873
biases = [-0.5, -0.25, 0.0, 0.25, 0.5]
spreads = [0.0, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
null_calibration = __import__('runpy').run_path(repo_root + '/src/r1/run.py')['run_step']('calibration_null', repo_root=repo_root, cache_dir=cache_dir, horizons=horizons, replicates=replicates, rhos=rhos, seed=seed, biases=biases, spreads=spreads)

Calibrated null gaussian_ar rho=0.0 H=48: hindsight gain=0.900%; independent transfer change=1.834%


Calibrated null gaussian_ar rho=0.9 H=48: hindsight gain=16.424%; independent transfer change=12.466%


Calibrated null gaussian_ar rho=0.0 H=480: hindsight gain=0.000%; independent transfer change=0.002%


Calibrated null gaussian_ar rho=0.9 H=480: hindsight gain=1.652%; independent transfer change=2.674%


Calibrated null log_price_rw rho=None H=48: hindsight gain=38.664%; independent transfer change=18.774%


Calibrated null log_price_rw rho=None H=480: hindsight gain=38.050%; independent transfer change=19.451%


{
  "cache": "/Users/mark/.cache/marktsf-research/r1/calibration_null_5bb30e5f51c2",
  "stage": "known_distribution_hindsight_null",
  "settings": [
    {
      "domain": "gaussian_ar",
      "rho": 0.0,
      "horizon": 48,
      "independent_path_pairs": 2048,
      "baseline_mean_pinball": 0.6163440936838778,
      "hindsight_path_improvement_fraction": 0.009002217789933598,
      "hindsight_half_improvement_fraction": 0.022957374763418525,
      "independent_transfer_path_degradation_fraction": 0.018336298508235795,
      "independent_transfer_half_degradation_fraction": 0.033608799291096794,
      "independent_transfer_path_difference_ci": {
        "mean": 0.011319131276973711,
        "ci95": [
          0.010382324603938217,
          0.012277477073666313
        ],
        "clusters": 2048
      },
      "independent_transfer_half_difference_ci": {
        "mean": 0.02074695779339096,
        "ci95": [
          0.019599159029168766,
          0.02195261114773798
        ],
  

## 当前证据与下一步

**尚未达到创新与SOTA目标。35个主步骤均已执行并保留输出。** 全部合同、负结果与近邻见 `RESEARCH.md` §8.1–8.12。

- 原子混合、持续时间—幅度校准、经典分布传播及单/多通道固定阈值年龄均未建立独立方法依据。
- 原生完整路径Toto常规组scaled geometric WQL=0.295114。第34步三种因果历史校准为0.295909/0.296826/0.298343，全部退化；long配置都因不足两条完整旧路径而保持原预测，不能解释为充分校准的长预测结果。
- 整段oracle可事后降到0.224612，但真实事件边界oracle没有胜过固定半段。所有oracle读取当前未来标签，不能作为可执行模型或SOTA成绩。
- 第35步以解析正确分位数作负对照：正价格随机游走仍出现约38%的整段oracle“改善”；把参数用于独立未来路径，误差却增加约19%。事后oracle空间不能当成可学习信号。
- 下一项机制门槛是预测起点可见条件下、跨时间/任务可复现的评分偏差。含原子分布必须使用严格/非严格概率不等式，不能把合法覆盖跳跃当作模型失准。这是已有统计诊断要求，不是新算法。

第34/35步分别在独立新kernel执行，前33步输出保留。全部逐案例分数、oracle选择参数、模拟路径对分数和源码快照只保存于仓库外。R1–R5研究对象与最终完整SOTA要求不变。